# Preparations

In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("full_survey.csv")
df

/var/folders/t5/8w8wlshx74lfgnc2wr0zy6dw0000gn/T/ipykernel_88795/135806008.py:1: DtypeWarning: Columns (0: scaleup_advanced_fee_flag, 1: scaleup_advanced_fee_desc, 2: scaleup_overpayment_flag, 3: scaleup_overpayment_desc, 4: visibility_overpayment_sureness, 5: visibility_nondelivery_sureness, 6: visibility_banking_sureness) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("full_survey.csv")


,id,start_date,bad_data,p_over,cstatus,screened_out,screener_advanced_fee,screener_nonpayment,screener_nondelivery,screener_extortion,...,hh1317,hh18ov,banking_lostmoney,scaleup_banking_confirmed,scaleup_banking_desc_lost_money_count_confirmed,scaleup_nondelivery_confirmed,scaleup_nonpayment_confirmed,scaleup_overpayment_confirmed,scaleup_extortion_confirmed,scaleup_advanced_fee_confirmed
0,Q1W5T0,2020-09-08T15:10:26Z,NaN,2,2,inconsistent,No,No,No,No,...,0,1,0,0,0.0,0,0,0,0,0
1,W4E4G8,2020-09-08T13:14:52Z,NaN,2,2,screener,No,No,No,No,...,0,1,0,0,0.0,0,0,0,0,0
2,S4F9G4,2020-09-08T11:45:40Z,NaN,2,2,screener,No,No,No,No,...,6,4,0,0,0.0,0,0,0,0,0
3,K1X4E3,2020-09-08T11:05:20Z,NaN,2,2,screener,No,No,No,No,...,0,3,0,0,0.0,0,0,0,0,0
4,H1D3D1,2020-09-08T10:40:07Z,NaN,2,2,screener,No,No,No,No,...,0,1,0,0,0.0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11948,W9B5D1,2020-07-21T16:52:52Z,NaN,1,1,no,No,No,No,No,...,0,1,0,0,0.0,0,0,0,0,0
11949,Y0D0C3,2020-07-21T16:49:06Z,NaN,1,1,no,No,No,No,No,...,0,4,0,0,0.0,1,0,0,0,0
11950,D2A8P7,2020-07-21T16:47:01Z,NaN,1,1,no,No,No,No,No,...,0,1,0,0,0.0,0,0,0,0,0
11951,F6L7Y9,2020-07-21T16:42:14Z,NaN,1,1,no,No,No,No,No,...,0,4,0,0,0.0,0,0,0,0,0


# Exploring the "lost money" category

In [8]:
print(df[df['banking_lostmoney'] == 1].shape)
banking = df[(df['screened_out'] == 'no') & (df['banking_lostmoney'] == 1)][['banking_money', 'weight2', 'gender', 'age4', 'racethnicity', 'educ', 'income', 'state']].copy()
banking['banking_money'] = pd.to_numeric(banking['banking_money'], errors='coerce')
banking['weight2'] = pd.to_numeric(banking['weight2'], errors='coerce')
banking = banking.dropna(subset=['banking_money', 'weight2'])
banking.shape, banking

(22, 142)


((22, 8),
        banking_money   weight2  gender  age4  racethnicity  educ  income state
 555             50.0  0.902404       2     2             1     3      11    FL
 7925            40.0  3.059766       1     1             1     2       9    MO
 9270           200.0  1.780869       2     4             4     3       7    CA
 9273           250.0  0.945291       1     2             1     4       9    CA
 10237           45.0  1.863764       1     1             1     4       9    OH
 10892         1000.0  1.104109       1     2             4     4       9    GA
 10929          200.0  1.050016       1     3             4     3      12    CA
 10945          200.0  6.260729       1     1             4     2       5    CA
 10973          200.0  0.506418       1     4             1     3      13    FL
 11031          500.0  2.786541       1     2             2     2       9    CT
 11039          450.0  2.346948       2     4             4     2       3    MO
 11088          200.0  0.71957

In [11]:
print(banking["banking_money"].sort_values(ascending=False).to_list())

[45000.0, 1000.0, 1000.0, 500.0, 450.0, 300.0, 300.0, 300.0, 250.0, 237.0, 200.0, 200.0, 200.0, 200.0, 200.0, 100.0, 100.0, 50.0, 45.0, 40.0, 0.0, 0.0]


# Reproducing Table 5 (Breen)

In [12]:
import numpy as np

crime_cols = {
    'banking': ('banking_lostmoney', 'banking_money'),
    'nondelivery': ('nondelivery', 'nondelivery_money'),
    'advanced_fee': ('advanced_fee', 'advanced_fee_money'),
    'nonpayment': ('nonpayment', 'nonpayment_money'),
    'extortion': ('extortion', 'extortion_money'),
    'overpayment': ('overpayment', 'overpayment_money'),
}

dfs = []
for crime, (flag_col, money_col) in crime_cols.items():
    subset = df[(df['screened_out'] == 'no') & (df[flag_col] == 1)][['weight2', money_col]].copy()
    subset['banking_money'] = pd.to_numeric(subset[money_col], errors='coerce')
    subset['weight2'] = pd.to_numeric(subset['weight2'], errors='coerce')
    subset = subset.dropna(subset=['banking_money', 'weight2'])
    subset['crime'] = crime
    subset = subset.rename(columns={money_col: 'money'})
    dfs.append(subset[['crime', 'money', 'weight2']])

incidents = pd.concat(dfs, ignore_index=True)
incidents.groupby('crime')['money'].describe()

def weighted_quantile(values, weights, quantiles):
    sorter = np.argsort(values)
    values = values.iloc[sorter].values
    weights = weights.iloc[sorter].values
    cumsum = np.cumsum(weights)
    cumsum /= cumsum[-1]
    return np.interp(quantiles, cumsum, values)

qs = [0.05, 0.10, 0.50, 0.90, 0.95]
results = {}
for crime, group in incidents.groupby('crime'):
    group = group.dropna(subset=['money', 'weight2'])
    results[crime] = weighted_quantile(group['money'], group['weight2'], qs)

pd.DataFrame(results, index=['q05', 'q10', 'median', 'q90', 'q95']).T

,q05,q10,median,q90,q95
advanced_fee,4.616560,13.381888,500.000000,3000.000000,3000.000000
banking,1.741283,32.335270,264.024861,1000.000000,14267.887688
extortion,54.355146,60.913984,300.000000,1445.460836,1489.970534
nondelivery,10.000000,15.000000,57.049508,300.000000,469.288327
nonpayment,9.702903,20.000000,98.858740,700.000000,875.258199
overpayment,35.000000,35.000000,100.000000,854.265000,1191.122163
